In [ ]:
import requests
import pandas as pd
import time
import os
from pathlib import Path
import boto3, json

if Path.cwd().name == "notebooks":
    os.chdir("..")

from src.config import load_config

CONFIG = load_config()

In [ ]:
import json

REGION = "eu-west-3"
ACCOUNT_ID = "844099234486"

ec2_session = boto3.Session(
    aws_access_key_id=os.getenv("ACCESS_KEY_ID"),
    aws_secret_access_key=os.getenv("SECRET_ACCESS_KEY"),
    region_name="eu-west-3",
)

ec2 = ec2_session.client("ec2")        # reutilise ta session EC2 existante
scheduler = ec2_session.client("scheduler")
iam = ec2_session.client("iam")

# 1. Recuperer les 2 instances
def get_instance_id(name_tag):
    resp = ec2.describe_instances(Filters=[{"Name": "tag:Name", "Values": [name_tag]}])
    return resp["Reservations"][0]["Instances"][0]["InstanceId"]

app_instance_id = get_instance_id("app-server")
mlflow_instance_id = get_instance_id("mlflow-server-v2")  # adapte le tag si different

instance_ids = [app_instance_id, mlflow_instance_id]
instance_arns = [f"arn:aws:ec2:{REGION}:{ACCOUNT_ID}:instance/{iid}" for iid in instance_ids]
print("app-server:", app_instance_id, "| mlflow-server:", mlflow_instance_id)

# 2. Role IAM pour le scheduler (start + stop, scope aux 2 instances)
SCHEDULER_EC2_ROLE_NAME = "nappecast-scheduler-ec2-role"  # remplace par le nom du role deja utilise si tu en as un

trust_policy = {
    "Version": "2012-10-17",
    "Statement": [{"Effect": "Allow", "Principal": {"Service": "scheduler.amazonaws.com"}, "Action": "sts:AssumeRole"}],
}
try:
    role = iam.create_role(RoleName=SCHEDULER_EC2_ROLE_NAME, AssumeRolePolicyDocument=json.dumps(trust_policy))
    scheduler_ec2_role_arn = role["Role"]["Arn"]
    print("Role cree :", scheduler_ec2_role_arn)
except iam.exceptions.EntityAlreadyExistsException:
    scheduler_ec2_role_arn = iam.get_role(RoleName=SCHEDULER_EC2_ROLE_NAME)["Role"]["Arn"]
    print("Role deja existant :", scheduler_ec2_role_arn)

ec2_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Action": ["ec2:StartInstances", "ec2:StopInstances"],
            "Resource": instance_arns,
        }
    ],
}
iam.put_role_policy(
    RoleName=SCHEDULER_EC2_ROLE_NAME,
    PolicyName="nappecast-start-stop-instances",
    PolicyDocument=json.dumps(ec2_policy),
)

import time
time.sleep(10)  # propagation IAM

# 3. Creation / mise a jour des 2 regles
def upsert_schedule(name, cron, action, description):
    params = dict(
        Name=name,
        ScheduleExpression=cron,
        ScheduleExpressionTimezone="Europe/Paris",
        FlexibleTimeWindow={"Mode": "OFF"},
        Target={
            "Arn": f"arn:aws:scheduler:::aws-sdk:ec2:{action}",
            "RoleArn": scheduler_ec2_role_arn,
            "Input": json.dumps({"InstanceIds": instance_ids}),
        },
        Description=description,
    )
    try:
        scheduler.create_schedule(**params)
        print("Regle creee :", name)
    except scheduler.exceptions.ConflictException:
        scheduler.update_schedule(**params)
        print("Regle mise a jour :", name)


upsert_schedule(
    "nappecast-stop-instances",
    "cron(30 19 * * ? *)",
    "stopInstances",
    "Arret quotidien des 2 EC2 (app-server + mlflow-server) a 19h30 Paris",
)

upsert_schedule(
    "nappecast-start-instances",
    "cron(50 5 * * ? *)",
    "startInstances",
    "Demarrage quotidien des 2 EC2 (app-server + mlflow-server) a 5h50 Paris, avant le pipeline",
)